# Eval-determinism gate v6 - breadth across LIBERO suites

v5 ran clean on 2026-09-12 22:32 UTC and returned **CONFOUND CONFIRMED**: T1-T4 pass, **T5 fails
3/3 seeds**, controls reproduce. Full record in `01 Projects/Paper Choice 2026-09-12.md` section 15.

**v6 answers what v5 could not: is it one task, or is it LIBERO?**

## What changed

- **Three suites, one task each** - `libero_object` (the v5 task, kept as a **replication
  control**), `libero_spatial`, `libero_goal`. Task index 0 in each suite, resolved by name at
  runtime rather than hardcoded.
- **T2 and T5 only.** T1 (seeded reset reproducibility) and T4 (prefix alignment) are settled and
  re-running them buys nothing.
- 🔴 **T6 is new, and it is here because of a defect I found in v5's own record.** See below.
- Cells 1-3 (install, probe, run-probe) are **byte-identical to v5**, carried across unchanged.

## Why T6 exists - the fix was never tested against the disease

Section 15 recorded T3 as *"the one-line fix the paper recommends."* Checking it against the raw
output, **T3 was run with `burn=0`**. It showed that reseeding removes dependence on *step count
and history* - which was never the confound that fails. **Reseeding has never been tested against
the global-RNG burn that actually breaks the episode sequence.**

The T1 cross-check makes it very likely the fix works (T3 seed-11 = T1 seed-12, and T3 seed-12 =
T1 seed-13, so `seed(s+1)` reproduces a fresh seed-(s+1) env twice over). **But likely is not
shown, and a paper proposing a remedy has to have run the remedy against the disease.**

**T6: burn 1000 global draws AND reseed before the reset.** Predicted **PASS** - if it fails, the
recommended fix does not work and the paper's conclusion changes.

## Pre-registered predictions - write nothing here after running

| Test | Question | **Prediction** |
|---|---|---|
| **T2** | next episode after K=40 vs K=80 steps, no reseed | **PASS** on all three suites |
| **T5** | 1000 global `np.random` draws between episodes | **FAIL** on all three suites |
| **T6** | same burn, but `seed()` called before the reset | **PASS** on all three suites |
| control | `libero_object` T2/T5 hashes vs v5 | **reproduce v5 exactly** |

**The control is the strongest single line in this notebook.** If `libero_object`'s T2 hash is not
`cbb3470b4c447f68` / `c732927029d36125` / `c09097460e7814a5` and its T5 burned hash is not
`e78e0d1a180b3e6a` / `42bec853db81b54b` / `dfa9091d9724f85f`, then something about the environment
changed between runs and **every other number in this notebook is suspect.** It is checked
automatically and reported as `v5_control`.

**Verdict rule, fixed in advance:** `GENERAL` iff T5 fails on all three suites - the confound is a
property of LIBERO, not of one task. `TASK-SPECIFIC` iff T5 fails on some but not all. `NOT
REPRODUCED` iff T5 passes everywhere, which would contradict v5 and mean the v5 result was an
artifact. `INCONCLUSIVE` if any control disagrees with itself or the v5 control fails.

## Expected wall time

v5's cell 5 took **13.7 min for one task** running five tests. Counting environment constructions,
which dominate: v5 built ~36 envs per task (T1 6, T2 9, T3 6, T4 6, T5 9). v6 builds ~24 per task
(T2 9, T5 9, T6 6) - about **two thirds of v5's per-task cost, so roughly 9 min per task**.

**Three tasks: expect cell 5 to run about 27-30 minutes**, plus ~6-8 min of install in cell 1.
Free Colab CPU is enough; no GPU, no checkpoint. Budget 40 minutes and do not let the tab idle out.


## Cell 1 - uv, a Python 3.11 venv, the pinned stack, and the transitive deps

In [ ]:
import subprocess, sys, os

def sh(cmd, check=True):
    print(">>>", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print("\n".join(r.stdout.strip().splitlines()[-12:]), flush=True)
    if check and r.returncode != 0:
        raise RuntimeError("FAILED (%d): %s" % (r.returncode, cmd))
    return r.returncode

sh("%s -m pip install -q uv" % sys.executable)
sh("uv venv /content/venv --python 3.11")

PY  = "/content/venv/bin/python"
UVP = "uv pip install --python " + PY

sh(UVP + " 'numpy==1.26.4'")
sh(UVP + " 'robosuite==1.4.0' 'bddl==1.0.1' 'gym==0.25.2' 'mujoco<3.2' "
         "easydict opencv-python-headless")

# the transitive deps v2 dropped with --no-deps
sh(UVP + " termcolor future hydra-core h5py pillow matplotlib cloudpickle "
         "pyyaml imageio tqdm")

# libero.libero.benchmark imports torch; CPU build only
sh(UVP + " torch --index-url https://download.pytorch.org/whl/cpu")

if not os.path.isdir("/content/LIBERO"):
    sh("git clone --depth 1 https://github.com/Lifelong-Robot-Learning/LIBERO.git /content/LIBERO")

# performed for completeness; nothing below depends on it succeeding
sh(UVP + " -e /content/LIBERO --no-deps", check=False)

# LIBERO first-run prompt: pre-seed config so no input() is needed
os.makedirs(os.path.expanduser("~/.libero"), exist_ok=True)
print("\ninstall step finished - the probe decides whether it worked")

## Cell 2 - write the probe (import-repair loop + resolved versions)

In [ ]:
%%writefile /content/probe.py
"""Verify the venv can actually import LIBERO, repairing missing transitive deps as it goes."""
import sys, subprocess, importlib
sys.path.insert(0, "/content/LIBERO")

VENV = "/content/venv/bin/python"

# The failing MODULE name is often not the PyPI PACKAGE name.
ALIAS = {
    "cv2": "opencv-python-headless", "PIL": "pillow", "yaml": "pyyaml",
    "skimage": "scikit-image", "sklearn": "scikit-learn", "attr": "attrs",
    "dateutil": "python-dateutil", "google": "protobuf", "pkg_resources": "setuptools",
    "mpl_toolkits": "matplotlib", "OpenGL": "pyopengl", "egl_probe": "egl-probe",
    "Cython": "cython", "IPython": "ipython",
}
NEVER_INSTALL = {"libero"}          # on PYTHONPATH, never a PyPI package here

def uv_install(pkg):
    r = subprocess.run(["uv", "pip", "install", "--python", VENV, pkg],
                       text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    ok = r.returncode == 0
    print("      uv pip install %s -> %s" % (pkg, "ok" if ok else "FAILED"), flush=True)
    if not ok:
        for line in r.stdout.strip().splitlines()[-4:]:
            print("      " + line, flush=True)
    return ok

TARGETS = ["libero.libero.envs", "libero.libero.benchmark"]
tried, MAX = set(), 15
green = False

print("== import-repair loop ==", flush=True)
for rnd in range(1, MAX + 1):
    try:
        for t in TARGETS:
            importlib.import_module(t)
        print("  round %d: all targets import cleanly" % rnd, flush=True)
        green = True
        break
    except ModuleNotFoundError as e:
        missing = (e.name or "").split(".")[0]
        print("  round %d: missing '%s'" % (rnd, missing), flush=True)
        if not missing or missing in NEVER_INSTALL:
            print("  STOP: '%s' is not installable here (expected on "
                  "PYTHONPATH=/content/LIBERO). Check the clone." % missing, flush=True)
            sys.exit(2)
        pkg = ALIAS.get(missing, missing)
        if pkg in tried:
            print("  STOP: already installed '%s' and '%s' still will not import."
                  % (pkg, missing), flush=True)
            sys.exit(3)
        tried.add(pkg)
        uv_install(pkg)
        for mod in [m for m in sys.modules if m.startswith("libero")]:
            sys.modules.pop(mod, None)
    except Exception as e:
        print("  round %d: non-import error %s: %s" % (rnd, type(e).__name__, e), flush=True)
        raise

if not green:
    print("  STOP: still failing after %d rounds. Installed: %s" % (MAX, sorted(tried)), flush=True)
    sys.exit(4)

print("\n== resolved versions, from inside the venv ==", flush=True)
print("%-12s %s" % ("python", sys.version.split()[0]))
for m in ("numpy", "robosuite", "mujoco", "bddl", "gym", "torch", "libero"):
    try:
        mod = importlib.import_module(m)
        extra = ("  <- " + str(getattr(mod, "__file__", "?"))) if m == "libero" else ""
        print("%-12s %s%s" % (m, getattr(mod, "__version__", "installed (no __version__)"), extra))
    except Exception as e:
        print("%-12s IMPORT FAILED: %s: %s" % (m, type(e).__name__, e))

if tried:
    print("\nrepaired by installing: %s" % sorted(tried))
    print("ADD THESE TO CELL 1 so the next run does not need the loop.")
print("\nPROBE OK - safe to run the gate")


## Cell 3 - run the probe

**If this does not end in `PROBE OK`, stop.** Cells 4-5 would
produce a number about a stack that cannot import LIBERO, which is exactly how v1 and v2
each wasted a run.

In [ ]:
import subprocess, os
env = dict(os.environ, PYTHONPATH="/content/LIBERO", MPLBACKEND="Agg")
p = subprocess.run(["/content/venv/bin/python", "/content/probe.py"],
                   env=env, text=True, input="N\n", stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(p.stdout)
print("probe exit code:", p.returncode)
assert p.returncode == 0, "probe failed - do not run the gate; paste the output above"

## Cell 4 - write the test script

## Cell 4 - write the breadth test script (T2, T5, T6 x 3 suites)

In [ ]:
%%writefile /content/gate6.py
"""v6: T2, T5 and T6 across three LIBERO suites. No policy, no checkpoint, no GPU."""
import os, sys, hashlib, json, datetime
sys.path.insert(0, "/content/LIBERO")

BACKEND = os.environ.get("MUJOCO_GL", "egl")
import numpy as np

SUITES       = ["libero_object", "libero_spatial", "libero_goal"]
SEEDS        = [11, 12, 13]
K_SHORT, K_LONG = 40, 80
GLOBAL_DRAWS = 1000

# v5's libero_object hashes, for the replication control
V5 = {
    "T2": {11: "cbb3470b4c447f68", 12: "c732927029d36125", 13: "c09097460e7814a5"},
    "T5_burned": {11: "e78e0d1a180b3e6a", 12: "42bec853db81b54b", 13: "dfa9091d9724f85f"},
}

from libero.libero import benchmark, get_libero_path
from libero.libero.envs import OffScreenRenderEnv

def bddl_for(suite):
    b = benchmark.get_benchmark_dict()[suite]()
    t = b.get_task(0)
    return os.path.join(get_libero_path("bddl_files"), t.problem_folder, t.bddl_file), t.name

def key(env):
    s = np.asarray(env.get_sim_state(), dtype=np.float64)
    return hashlib.sha256(np.ascontiguousarray(s).tobytes()).hexdigest()[:16]

def acts(k, tag, env):
    rs = np.random.RandomState(1234 if tag == "armA" else 5678)
    return [rs.uniform(-0.2, 0.2, size=env.env.action_dim) for _ in range(k)]

def roll_reset(bddl, s, k, tag, reseed=None, burn=0):
    e = OffScreenRenderEnv(bddl_file_name=bddl, camera_heights=128, camera_widths=128)
    e.seed(int(s)); e.reset()
    for a in acts(k, tag, e):
        e.step(a)
    for _ in range(burn):
        np.random.random()
    if reseed is not None:
        e.seed(int(reseed))
    e.reset()
    kk = key(e); e.close(); return kk

R = {"suites": SUITES, "seeds": SEEDS, "k_short": K_SHORT, "k_long": K_LONG,
     "global_draws": GLOBAL_DRAWS, "mujoco_gl": BACKEND,
     "predictions": {"T2": "PASS all suites", "T5": "FAIL all suites",
                     "T6": "PASS all suites", "v5_control": "reproduce exactly"},
     "per_suite": {}}

for suite in SUITES:
    bddl, tname = bddl_for(suite)
    print("\n" + "=" * 68, flush=True)
    print("SUITE %s  task0=%s" % (suite, tname), flush=True)
    print("=" * 68, flush=True)
    S = {"task": tname, "T2": {}, "T5": {}, "T6": {}}

    print("-- T2 step-count dependence", flush=True)
    for s in SEEDS:
        sh = roll_reset(bddl, s, K_SHORT, "armA")
        lo = roll_reset(bddl, s, K_LONG,  "armB")
        ct = roll_reset(bddl, s, K_SHORT, "armA")
        S["T2"][s] = {"short": sh, "long": lo, "control": ct,
                      "arms_match": sh == lo, "control_match": sh == ct}
        print("   seed %s: %s vs %s -> %s ctrl=%s"
              % (s, sh, lo, "MATCH" if sh == lo else "DIFFER",
                 "ok" if sh == ct else "UNSTABLE"), flush=True)

    print("-- T5 global-RNG leakage", flush=True)
    for s in SEEDS:
        base = roll_reset(bddl, s, K_SHORT, "armA", burn=0)
        burn = roll_reset(bddl, s, K_SHORT, "armA", burn=GLOBAL_DRAWS)
        ctrl = roll_reset(bddl, s, K_SHORT, "armA", burn=0)
        S["T5"][s] = {"no_burn": base, "burned": burn, "control": ctrl,
                      "match": base == burn, "control_match": base == ctrl}
        print("   seed %s: %s -> %s : %s ctrl=%s"
              % (s, base, burn,
                 "MATCH (no leak)" if base == burn else "DIFFER (LEAK)",
                 "ok" if base == ctrl else "UNSTABLE"), flush=True)

    print("-- T6 does reseeding survive the burn? (the fix, vs the disease)", flush=True)
    for s in SEEDS:
        a = roll_reset(bddl, s, K_SHORT, "armA", reseed=s + 1, burn=0)
        b = roll_reset(bddl, s, K_SHORT, "armA", reseed=s + 1, burn=GLOBAL_DRAWS)
        S["T6"][s] = {"no_burn": a, "burned": b, "match": a == b}
        print("   seed %s: %s vs %s -> %s"
              % (s, a, b, "MATCH (fix holds)" if a == b else "DIFFER (FIX FAILS)"), flush=True)

    S["T2_pass"] = all(v["arms_match"] for v in S["T2"].values())
    S["T2_control_ok"] = all(v["control_match"] for v in S["T2"].values())
    S["T5_pass"] = all(v["match"] for v in S["T5"].values())
    S["T5_control_ok"] = all(v["control_match"] for v in S["T5"].values())
    S["T6_pass"] = all(v["match"] for v in S["T6"].values())
    R["per_suite"][suite] = S
    print("   => T2_pass=%s T5_pass=%s T6_pass=%s"
          % (S["T2_pass"], S["T5_pass"], S["T6_pass"]), flush=True)

# replication control against v5
ctl = {}
if "libero_object" in R["per_suite"]:
    o = R["per_suite"]["libero_object"]
    ctl = {str(s): {"t2_now": o["T2"][s]["short"], "t2_v5": V5["T2"][s],
                    "t2_match": o["T2"][s]["short"] == V5["T2"][s],
                    "t5_now": o["T5"][s]["burned"], "t5_v5": V5["T5_burned"][s],
                    "t5_match": o["T5"][s]["burned"] == V5["T5_burned"][s]}
           for s in SEEDS}
V5_CONTROL_OK = bool(ctl) and all(v["t2_match"] and v["t5_match"] for v in ctl.values())
R["v5_control"] = {"detail": ctl, "reproduced": V5_CONTROL_OK}

print("\n== v5 replication control (libero_object) ==", flush=True)
for s in SEEDS:
    d = ctl[str(s)]
    print("   seed %s: T2 %s (%s) | T5 %s (%s)"
          % (s, "MATCH" if d["t2_match"] else "MISMATCH", d["t2_now"],
             "MATCH" if d["t5_match"] else "MISMATCH", d["t5_now"]), flush=True)

controls_ok = V5_CONTROL_OK and all(
    S["T2_control_ok"] and S["T5_control_ok"] for S in R["per_suite"].values())
t5_fail = [k for k, S in R["per_suite"].items() if not S["T5_pass"]]

if not controls_ok:
    verdict = "INCONCLUSIVE"
    reading = ("A control disagreed with itself, or libero_object did not reproduce v5. "
               "Every other number here is suspect.")
elif len(t5_fail) == len(SUITES):
    verdict = "GENERAL"
    reading = ("Global-RNG consumption shifts the episode sequence on all %d suites. "
               "The confound is a property of LIBERO, not of one task." % len(SUITES))
elif t5_fail:
    verdict = "TASK-SPECIFIC"
    reading = "Leakage appears on %s but not on the others. Narrow the claim." % ", ".join(t5_fail)
else:
    verdict = "NOT REPRODUCED"
    reading = "T5 passed everywhere, contradicting v5. Treat the v5 result as an artifact until explained."

import importlib
ver = {"python": sys.version.split()[0]}
for m in ("numpy", "robosuite", "mujoco", "gym", "torch"):
    try:
        ver[m] = getattr(importlib.import_module(m), "__version__", "installed-no-__version__")
    except Exception as e:
        ver[m] = "IMPORT FAILED: %s" % type(e).__name__

t6_all = all(S["T6_pass"] for S in R["per_suite"].values())
R.update({"timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
          "versions": ver, "verdict": verdict, "reading": reading,
          "t5_failed_on": t5_fail, "T6_pass_all": t6_all, "controls_ok": controls_ok})

with open("/content/det_result_v6.json", "w") as f:
    json.dump(R, f, indent=2, default=str)

print("\n" + "=" * 68)
print(json.dumps({k: R[k] for k in
                  ["timestamp_utc", "verdict", "reading", "t5_failed_on", "T6_pass_all",
                   "controls_ok", "v5_control", "versions", "predictions"]}, indent=2))
print("=" * 68)
print("full record: /content/det_result_v6.json")
print("T6 is the fix tested against the disease. If T6_pass_all is false, the")
print("paper's recommended remedy does not work and the conclusion changes.")


## Cell 5 - run it

Expect ~27-30 min. `input='N'` and `MPLBACKEND=Agg` are the two
environment fixes v4 and v5 needed; both are kept.

In [ ]:
import subprocess, os, time

def run_gate(backend):
    env = dict(os.environ, MUJOCO_GL=backend, PYOPENGL_PLATFORM=backend,
               PYTHONPATH="/content/LIBERO", MPLBACKEND="Agg")
    p = subprocess.run(["/content/venv/bin/python", "/content/gate6.py"],
                       env=env, input="N\n", text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout)
    return p.returncode

t0 = time.time()
rc = run_gate("egl")
if rc != 0:
    print("\nEGL failed (exit %d). Installing OSMesa and retrying once." % rc)
    subprocess.run("apt-get -qq install -y libosmesa6-dev > /dev/null 2>&1", shell=True)
    rc = run_gate("osmesa")

print("\nexit code: %d | elapsed %.1f min" % (rc, (time.time() - t0) / 60))
print("PASTE THE JSON BLOCK ABOVE BACK for Paper Choice 2026-09-12.md section 18" if rc == 0
      else "Failed on both backends. Paste the traceback; this is not a result.")